# Manipulating DataFrames with String Operations

When processing raw data, string columns frequently house concatenated values that demand segmentation into granular columns. Pandas delivers robust toolsets via the `.str` namespace alongside mapping methods such as `.apply()`.

Topics covered:
- Function execution via `.apply()`
- Substring extraction using Regular Expressions (`.str.extract()`) 
- Cleaning overlapping metadata
- Typecasting temporal strings (`pd.to_datetime()`)

In [1]:
# Import pandas
import pandas as pd

## Functional Programming via Apply

Let's assume we want to isolate "First" and "Last" names from a single presidential name column. We could accomplish this by routing every row into a custom parsing function via the `.apply(func, axis)` routine.

In [2]:
# Load presidents dataset
df = pd.read_csv("datasets/presidents.csv")
df.head()

,#,President,Born,Age atstart of presidency,Age atend of presidency,Post-presidencytimespan,Died,Age
0,1,George Washington,"Feb 22, 1732[a]","57 years, 67 daysApr 30, 1789","65 years, 10 daysMar 4, 1797","2 years, 285 days","Dec 14, 1799","67 years, 295 days"
1,2,John Adams,"Oct 30, 1735[a]","61 years, 125 daysMar 4, 1797","65 years, 125 daysMar 4, 1801","25 years, 122 days","Jul 4, 1826","90 years, 247 days"
2,3,Thomas Jefferson,"Apr 13, 1743[a]","57 years, 325 daysMar 4, 1801","65 years, 325 daysMar 4, 1809","17 years, 122 days","Jul 4, 1826","83 years, 82 days"
3,4,James Madison,"Mar 16, 1751[a]","57 years, 353 daysMar 4, 1809","65 years, 353 daysMar 4, 1817","19 years, 116 days","Jun 28, 1836","85 years, 104 days"
4,5,James Monroe,"Apr 28, 1758","58 years, 310 daysMar 4, 1817","66 years, 310 daysMar 4, 1825","6 years, 122 days","Jul 4, 1831","73 years, 67 days"


In [3]:
# Write custom row processor algorithm
def splitname(row):
    row["First"] = row["President"].split(" ")[0]
    row["Last"] = row["President"].split(" ")[-1]
    return row

# Apply logic iteratively against the 'columns' axis (every row)
df = df.apply(splitname, axis="columns")
df.head()

,#,President,Born,Age atstart of presidency,Age atend of presidency,Post-presidencytimespan,Died,Age,First,Last
0,1,George Washington,"Feb 22, 1732[a]","57 years, 67 daysApr 30, 1789","65 years, 10 daysMar 4, 1797","2 years, 285 days","Dec 14, 1799","67 years, 295 days",George,Washington
1,2,John Adams,"Oct 30, 1735[a]","61 years, 125 daysMar 4, 1797","65 years, 125 daysMar 4, 1801","25 years, 122 days","Jul 4, 1826","90 years, 247 days",John,Adams
2,3,Thomas Jefferson,"Apr 13, 1743[a]","57 years, 325 daysMar 4, 1801","65 years, 325 daysMar 4, 1809","17 years, 122 days","Jul 4, 1826","83 years, 82 days",Thomas,Jefferson
3,4,James Madison,"Mar 16, 1751[a]","57 years, 353 daysMar 4, 1809","65 years, 353 daysMar 4, 1817","19 years, 116 days","Jun 28, 1836","85 years, 104 days",James,Madison
4,5,James Monroe,"Apr 28, 1758","58 years, 310 daysMar 4, 1817","66 years, 310 daysMar 4, 1825","6 years, 122 days","Jul 4, 1831","73 years, 67 days",James,Monroe


In [4]:
# Clean up our variables before the next demonstration
del(df["First"])
del(df["Last"])
df.head()

,#,President,Born,Age atstart of presidency,Age atend of presidency,Post-presidencytimespan,Died,Age
0,1,George Washington,"Feb 22, 1732[a]","57 years, 67 daysApr 30, 1789","65 years, 10 daysMar 4, 1797","2 years, 285 days","Dec 14, 1799","67 years, 295 days"
1,2,John Adams,"Oct 30, 1735[a]","61 years, 125 daysMar 4, 1797","65 years, 125 daysMar 4, 1801","25 years, 122 days","Jul 4, 1826","90 years, 247 days"
2,3,Thomas Jefferson,"Apr 13, 1743[a]","57 years, 325 daysMar 4, 1801","65 years, 325 daysMar 4, 1809","17 years, 122 days","Jul 4, 1826","83 years, 82 days"
3,4,James Madison,"Mar 16, 1751[a]","57 years, 353 daysMar 4, 1809","65 years, 353 daysMar 4, 1817","19 years, 116 days","Jun 28, 1836","85 years, 104 days"
4,5,James Monroe,"Apr 28, 1758","58 years, 310 daysMar 4, 1817","66 years, 310 daysMar 4, 1825","6 years, 122 days","Jul 4, 1831","73 years, 67 days"


## Vectorized Regex Extractions (`str.extract`)

Instead of writing slow Python logic, using the vectorized `.str.extract()` command accompanied by capturing groups yields cleaner mechanics. Regex parsing captures any pattern isolated inside unescaped parenthesis: `()`

In [5]:
# Regex capture groups: 
# 1. (^\w*) - captures word characters from string head
# 2. (\w*$) - captures word characters immediately preceeding string termination
pattern=r"(^[\w]*)(?:.* )([\w]*$)"

# .extract creates a new dataframe built from the generated capture groups
df["President"].str.extract(pattern).head()

,0,1
0,George,Washington
1,John,Adams
2,Thomas,Jefferson
3,James,Madison
4,James,Monroe


### Naming Regexp Groups for Automatic Column Generation

Using python's named group mechanics `(?P<name>)` automatically outputs columns appropriately named.

In [6]:
# Inject naming specifiers ?P<First> and ?P<Last>
pattern=r"(?P<First>^[\w]*)(?:.* )(?P<Last>[\w]*$)"

names = df["President"].str.extract(pattern)
names.head()

,First,Last
0,George,Washington
1,John,Adams
2,Thomas,Jefferson
3,James,Madison
4,James,Monroe


In [7]:
# Transplant the generated vectors directly into the working set
df["First"] = names["First"]
df["Last"] = names["Last"]
df.head()

,#,President,Born,Age atstart of presidency,Age atend of presidency,Post-presidencytimespan,Died,Age,First,Last
0,1,George Washington,"Feb 22, 1732[a]","57 years, 67 daysApr 30, 1789","65 years, 10 daysMar 4, 1797","2 years, 285 days","Dec 14, 1799","67 years, 295 days",George,Washington
1,2,John Adams,"Oct 30, 1735[a]","61 years, 125 daysMar 4, 1797","65 years, 125 daysMar 4, 1801","25 years, 122 days","Jul 4, 1826","90 years, 247 days",John,Adams
2,3,Thomas Jefferson,"Apr 13, 1743[a]","57 years, 325 daysMar 4, 1801","65 years, 325 daysMar 4, 1809","17 years, 122 days","Jul 4, 1826","83 years, 82 days",Thomas,Jefferson
3,4,James Madison,"Mar 16, 1751[a]","57 years, 353 daysMar 4, 1809","65 years, 353 daysMar 4, 1817","19 years, 116 days","Jun 28, 1836","85 years, 104 days",James,Madison
4,5,James Monroe,"Apr 28, 1758","58 years, 310 daysMar 4, 1817","66 years, 310 daysMar 4, 1825","6 years, 122 days","Jul 4, 1831","73 years, 67 days",James,Monroe


## Typecasting Strings into Temporal Metadata

Raw data sometimes taints date vectors with references. For example, presidential datasets typically attach footnotes (e.g. `[a]`) to 'Born' timelines. 

We extract just the chronological payload before typecasting into a true `Datetime` structure with `pd.to_datetime()`.

In [8]:
# Extract exactly: {3 letters} {1-2 letters}, {4 letters}
df["Born"] = df["Born"].str.extract(r"([\w]{3} [\w]{1,2}, [\w]{4})")
df["Born"].head()

0    Feb 22, 1732
1    Oct 30, 1735
2    Apr 13, 1743
3    Mar 16, 1751
4    Apr 28, 1758
Name: Born, dtype: object

In [9]:
# Now that interference is sanitized, transition strings structurally to DateTime components
df["Born"] = pd.to_datetime(df["Born"])
df["Born"].head()

0   1732-02-22
1   1735-10-30
2   1743-04-13
3   1751-03-16
4   1758-04-28
Name: Born, dtype: datetime64[ns]